# Model Serialization and Versioning

This notebook demonstrates how to save, load, and version machine learning models for deployment.

## Topics Covered
1. Train a model on the Iris dataset
2. Save with `pickle` and `joblib`, compare file sizes
3. Load and verify predictions match
4. Create metadata JSON alongside the model
5. Best practices for model versioning

In [ ]:
import numpy as np
import pickle
import joblib
import json
import os
import time
from datetime import datetime
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Train the Model

In [ ]:
# Load data
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {iris.feature_names}")
print(f"Classes: {list(iris.target_names)}")

# Create a pipeline with scaling and classifier
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train
pipeline.fit(X_train, y_train)

# Evaluate
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
cv_scores = cross_val_score(pipeline, X, y, cv=5)

print(f"\nTest accuracy: {accuracy:.4f}")
print(f"CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

## 2. Save with Pickle and Joblib

In [ ]:
# Create output directory
os.makedirs('model_artifacts', exist_ok=True)

# Save with pickle
pickle_path = 'model_artifacts/model.pkl'
start = time.time()
with open(pickle_path, 'wb') as f:
    pickle.dump(pipeline, f)
pickle_save_time = time.time() - start
pickle_size = os.path.getsize(pickle_path)

# Save with joblib
joblib_path = 'model_artifacts/model.joblib'
start = time.time()
joblib.dump(pipeline, joblib_path)
joblib_save_time = time.time() - start
joblib_size = os.path.getsize(joblib_path)

# Save with joblib (compressed)
joblib_compressed_path = 'model_artifacts/model_compressed.joblib'
start = time.time()
joblib.dump(pipeline, joblib_compressed_path, compress=3)
joblib_compressed_save_time = time.time() - start
joblib_compressed_size = os.path.getsize(joblib_compressed_path)

print("File Size Comparison:")
print(f"  pickle:              {pickle_size:>8,} bytes  (save: {pickle_save_time:.4f}s)")
print(f"  joblib:              {joblib_size:>8,} bytes  (save: {joblib_save_time:.4f}s)")
print(f"  joblib (compressed): {joblib_compressed_size:>8,} bytes  (save: {joblib_compressed_save_time:.4f}s)")
print(f"\nCompression ratio: {pickle_size / joblib_compressed_size:.2f}x")

## 3. Load and Verify Predictions Match

In [ ]:
# Load with pickle
start = time.time()
with open(pickle_path, 'rb') as f:
    model_pickle = pickle.load(f)
pickle_load_time = time.time() - start

# Load with joblib
start = time.time()
model_joblib = joblib.load(joblib_path)
joblib_load_time = time.time() - start

# Load compressed joblib
start = time.time()
model_joblib_compressed = joblib.load(joblib_compressed_path)
joblib_compressed_load_time = time.time() - start

# Verify predictions match
pred_original = pipeline.predict(X_test)
pred_pickle = model_pickle.predict(X_test)
pred_joblib = model_joblib.predict(X_test)
pred_compressed = model_joblib_compressed.predict(X_test)

print("Prediction Verification:")
print(f"  pickle matches original:     {np.array_equal(pred_original, pred_pickle)}")
print(f"  joblib matches original:     {np.array_equal(pred_original, pred_joblib)}")
print(f"  compressed matches original: {np.array_equal(pred_original, pred_compressed)}")

print(f"\nLoad Times:")
print(f"  pickle:              {pickle_load_time:.4f}s")
print(f"  joblib:              {joblib_load_time:.4f}s")
print(f"  joblib (compressed): {joblib_compressed_load_time:.4f}s")

## 4. Create Model Metadata

In [ ]:
metadata = {
    "model_name": "iris_classifier",
    "model_version": "1.0.0",
    "created_at": datetime.now().isoformat(),
    "framework": "scikit-learn",
    "algorithm": "RandomForestClassifier",
    "pipeline_steps": [type(step[1]).__name__ for step in pipeline.steps],
    "hyperparameters": {
        "n_estimators": 100,
        "random_state": 42
    },
    "dataset": {
        "name": "Iris",
        "n_samples": len(X),
        "n_features": X.shape[1],
        "feature_names": list(iris.feature_names),
        "target_names": list(iris.target_names),
        "n_classes": len(iris.target_names)
    },
    "metrics": {
        "test_accuracy": float(accuracy),
        "cv_accuracy_mean": float(cv_scores.mean()),
        "cv_accuracy_std": float(cv_scores.std()),
        "cv_folds": 5
    },
    "training": {
        "train_samples": len(X_train),
        "test_samples": len(X_test),
        "test_size": 0.2
    },
    "artifacts": {
        "pickle_file": "model.pkl",
        "pickle_size_bytes": pickle_size,
        "joblib_file": "model.joblib",
        "joblib_size_bytes": joblib_size
    },
    "input_schema": {
        "type": "array",
        "shape": [None, 4],
        "dtype": "float64",
        "description": "Array of shape (n_samples, 4) with sepal/petal length/width"
    },
    "output_schema": {
        "type": "array",
        "values": [0, 1, 2],
        "labels": list(iris.target_names),
        "description": "Predicted class index"
    }
}

# Save metadata
metadata_path = 'model_artifacts/metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print("Model metadata saved to model_artifacts/metadata.json")
print(json.dumps(metadata, indent=2))

## 5. Visualization: File Size Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# File size comparison
methods = ['pickle', 'joblib', 'joblib\n(compressed)']
sizes_kb = [pickle_size/1024, joblib_size/1024, joblib_compressed_size/1024]
colors = ['#3498db', '#2ecc71', '#e74c3c']

bars = axes[0].bar(methods, sizes_kb, color=colors, edgecolor='black', alpha=0.8)
for bar, size in zip(bars, sizes_kb):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{size:.1f} KB', ha='center', fontweight='bold')
axes[0].set_ylabel('File Size (KB)', fontsize=12)
axes[0].set_title('Model File Size Comparison', fontsize=14)

# Save/Load time comparison
save_times = [pickle_save_time*1000, joblib_save_time*1000, joblib_compressed_save_time*1000]
load_times = [pickle_load_time*1000, joblib_load_time*1000, joblib_compressed_load_time*1000]

x = np.arange(len(methods))
width = 0.35
bars1 = axes[1].bar(x - width/2, save_times, width, label='Save', color='steelblue', edgecolor='black')
bars2 = axes[1].bar(x + width/2, load_times, width, label='Load', color='coral', edgecolor='black')
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods)
axes[1].set_ylabel('Time (ms)', fontsize=12)
axes[1].set_title('Save/Load Time Comparison', fontsize=14)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Clean up artifacts
import shutil
if os.path.exists('model_artifacts'):
    shutil.rmtree('model_artifacts')
    print("Cleaned up model_artifacts/ directory.")

## Best Practices for Model Versioning

1. **Always save metadata** alongside the model - include training data info, metrics, hyperparameters, and timestamps.

2. **Use semantic versioning** (major.minor.patch):
   - Major: breaking changes (new features, different input schema)
   - Minor: improved model (retrained, better accuracy)
   - Patch: bug fixes, no model change

3. **joblib vs pickle**:
   - `joblib` is generally faster for models with large NumPy arrays
   - `joblib` supports built-in compression
   - `pickle` is in the standard library (no extra dependency)

4. **Store in a model registry** (MLflow, DVC, or cloud-specific services) for:
   - Version tracking
   - Lineage (which data/code produced this model)
   - Stage management (staging -> production -> archived)

5. **Security**: Never load pickled models from untrusted sources - pickle can execute arbitrary code.

6. **Reproducibility**: Pin your dependency versions and save the full environment spec.